# Strands Agents with AgentCore Memory (Long-term) using tools

## Overview
This notebook demonstrates how to implement long-term memory capabilities for conversational AI agents using Strands and AgentCore Memory. You'll learn how to extract and consolidate important information from short-term interactions, enabling an agent to recall key details across multiple conversation sessions over time.

## Tutorial Details
**Use Case:** Culinary Assistant with Persistent Memory

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Long term Conversational                                                         |
| Agent type          | Culinary Assistant                                                               |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Anthropic Claude Sonnet 3.7                                                      |
| Tutorial components | AgentCore 'User Preferences' Memory Extraction, Memory Tool for storing and retrieving Memory              |
| Example complexity  | Beginner                                                                     |

You'll learn to:
- Configure AgentCore Memory with extraction strategies for long-term retention
- Hydrate memory with previous conversation history
- Use long-term memory to deliver personalized experiences across conversation sessions
- Integrate Strands Agent Framework with the AgentCore Memory tool

## Scenario Context

In this tutorial, you'll step into the role of a Culinary Assistant designed to deliver highly personalized restaurant recommendations. By leveraging AgentCore Memory's long-term retention and automatic information extraction, the agent can remember user preferences—such as dietary choices and favorite cuisines—across multiple conversations. This persistent memory enables the agent to provide tailored suggestions and a seamless user experience, even as conversations span days or weeks. The scenario demonstrates how structured memory organization and configurable strategies empower conversational AI to move beyond short-term recall, creating truly engaging and context-aware interactions.


## Architecture

<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>


## Prerequisites

To execute this tutorial you will need:
- Python 3.10+
- AWS credentials with Amazon Bedrock AgentCore Memory permissions
- Amazon Bedrock AgentCore SDK
Let's get started by setting up our environment and creating our long-term memory resource with the appropriate extraction strategy!

## Environment set up
Let's begin importing all the necessary libraries and defining the clients to make this notebook work.

In [ ]:
!pip install -qr requirements-dev.txt
!pip install -qr requirements.txt


## Creating Memory with Long-Term Strategies

In this section, we'll create a memory resource configured with long-term memory capabilities. Unlike our previous short-term memory example, this implementation includes specific memory strategies that enable consolidated information retention.

In [ ]:
import os
import time
import time
from datetime import datetime
from botocore.exceptions import ClientError
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

memory_name = "CulinaryAssistant"
memory_id = None

region = os.getenv('AWS_REGION', 'us-east-1')
client = MemoryClient(region_name=region)

try:
    print("Creating Long-Term Memory...")

    # We use a more descriptive name for our long-term memory resource
    memory_name = memory_name

    # Create memory with user preference strategy
    memory = client.create_memory_and_wait(
        name=memory_name,
        description="Culinary Assistant Agent with long term memory",
        strategies=[{
                    StrategyType.USER_PREFERENCE.value: {
                        "name": "UserPreferences",
                        "description": "Captures user preferences",
                        "namespaces": ["user/{actorId}/preferences"]
                    }
                }],
        event_expiry_days=7,
        max_wait=300,
        poll_interval=10
    )

    memory_id = memory['id']
    print(f"Memory created successfully with ID: {memory_id}")
    
except ClientError as e:
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        # If memory already exists, retrieve its ID
        memories = client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        print(f"Memory already exists. Using existing memory ID: {memory_id}")
except Exception as e:
    # Handle any errors during memory creation
    logger.info(f"❌ ERROR: {e}")
    import traceback
    traceback.print_exc()
    # Cleanup on error - delete the memory if it was partially created
    if memory_id:
        try:
            client.delete_memory_and_wait(memory_id=memory_id)
            print(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            print(f"Failed to clean up memory: {cleanup_error}")

### Understanding Long-Term Memory Strategies

The key difference in this memory creation is the addition of a **memory strategy**. Let's break down the components:

#### 1. User Preference Memory Strategy

This strategy automatically identifies and extracts user preferences from conversations:

```python
"userPreferenceMemoryStrategy": {
    "name": "UserPreferences",
    "description": "Captures user preferences",
    "namespaces": ["user/{actorId}/preferences"]
}
```

#### 2. Memory Namespaces

The `namespaces` parameter defines where extracted information is stored:

```python
"namespaces": ["user/{actorId}/preferences"]
```
This memory strategy creates a more sophisticated memory system that doesn't just remember conversations, but actually understands and organizes the important information within those conversations for future use.

## Saving Previous Conversations to Memory

In this section, we'll demonstrate how to hydrate the short-term memory, which automatically triggers the long-term memory extraction process behind the scenes.

### Hydrating Short-Term Memory

When we save conversations to a memory resource configured with extraction strategies, the system automatically processes this information for long-term retention without requiring additional code.

In [ ]:
print("\nHydrating short term memory with previous conversations...")

# actor_id = f"user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
# session_id = f"foodie-{datetime.now().strftime('%Y%m%d%H%M%S')}"
actor_id = "user-20251103170743"
session_id = "foodie-example-session-20251103170743"
namespace = f"user/{actor_id}/preferences"

previous_messages = [
    ("Hi, I'm John", "USER"),
    ("Hi John, how can I help you with food recommendations today?", "ASSISTANT"),
    ("I'm looking for some vegetarian dishes to try this weekend.", "USER"),
    ("That sounds great! I'd be happy to help with vegetarian recommendations. Do you have any specific ingredients or cuisine types you prefer?", "ASSISTANT"),
    ("Yes, I really like tofu and fresh vegetables in my dishes", "USER"),
    ("Perfect! Tofu and fresh vegetables make for excellent vegetarian meals. I can suggest some stir-fries, Buddha bowls, or tofu curries. Do you have any other preferences?", "ASSISTANT"),
    ("I also really enjoy Italian cuisine. I love pasta dishes and would like them to be vegetarian-friendly.", "USER"),
    ("Excellent! Italian cuisine has wonderful vegetarian options. I can recommend pasta primavera, mushroom risotto, eggplant parmesan, or penne arrabbiata. The combination of Italian flavors with vegetarian ingredients creates delicious meals!", "ASSISTANT"),
    ("I spent 2 hours looking through cookbooks but couldn't find inspiring vegetarian Italian recipes", "USER"),
    ("I'm sorry you had trouble finding inspiring recipes! Let me help you with some creative vegetarian Italian dishes. How about stuffed bell peppers with Italian herbs and rice, spinach and ricotta cannelloni, or a Mediterranean vegetable lasagna?", "ASSISTANT"),
    ("Hey, I appreciate food assistants with good taste", "USER"),
    ("Ha! I definitely try to bring good taste to the table! Speaking of which, shall we explore some more vegetarian Italian recipes that might inspire you?", "ASSISTANT")
]

# Save the conversation history to long-term memory
initial = client.create_event(
    memory_id=memory_id,
    actor_id=actor_id,
    session_id=session_id,
    messages=previous_messages,
)
print("✓ Conversation saved in short term memory")

Let's make sure the event containing the conversation messages was stored correctly.

In [ ]:
events = client.list_events(
    memory_id=memory_id,
    actor_id=actor_id,
    session_id=session_id,
    max_results=5
)
events

### What Happens Behind the Scenes

After the `create_event` call, the following occurs automatically:

1. **Short-Term Storage**: The complete conversation is saved in raw form
2. **Extraction Trigger**: The memory system detects that this memory has the UserPreference strategy configured
3. **Background Processing**: Without any additional code, the system:
   - Analyzes the conversation for preference indicators
   - Identifies statements like "I'm vegetarian" and "I really enjoy Italian cuisine"
   - Extracts these preferences into structured data
4. **Long-Term Consolidation**: The extracted preferences are saved in the configured namespace (`user/{actorId}/preferences`)

Extraction and consolidation happen automatically - we only need to mantain a conversation with the agent or hydrate the short-term memory, and the strategies we configured during memory creation take care of the rest.

This automatic process ensures that important information is preserved in long-term memory even after the short-term conversation records expire.


## Retrieving Long-Term Memories

In this section, we'll explore how to access the extracted preferences that have been stored in long-term memory. Unlike short-term memory retrieval which focuses on conversation turns, long-term memory retrieval focuses on accessing structured information that has been extracted and consolidated.

### Accessing User Preferences from Long-Term Memory

To retrieve information from long-term memory, we use the namespace structure defined during memory creation:


In [ ]:
# Adding a 30s wait to ensure the memory extraction has time to process the event
time.sleep(30)

try:
    # Query the memory system for food preferences
    food_preferences = client.retrieve_memories(
        memory_id=memory_id,
        namespace=namespace,
        query="food preferences",
        top_k=3  # Return up to 3 most relevant results
    )

    if food_preferences:
        print(f"Retrieved {len(food_preferences)} relevant preference records:")
        for i, record in enumerate(food_preferences):
            print(f"\nMemory {i+1}:")
            print(f"- Content: {record.get('content', 'Not specified')}")
    else:
        print("No matching preference records found.")

except Exception as e:
    print(f"Error retrieving preference records: {e}")

This method enables the retrieval of relevant memories when needed. Now we learned the basics let's build up our agent!

## Create Strands Agent

In [ ]:
%%writefile strands_claude_ltm.py
import os
from strands import Agent
from strands.models import BedrockModel
from strands.telemetry import StrandsTelemetry
from strands_tools.agent_core_memory import AgentCoreMemoryToolProvider
from bedrock_agentcore.runtime import BedrockAgentCoreApp, RequestContext


MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
SYSTEM_PROMPT = """You are the Culinary Assistant, a sophisticated restaurant recommendation assistant.
PURPOSE:
- Help users discover restaurants based on their preferences
- Remember user preferences throughout the conversation
- Provide personalized dining recommendations

You have access to a Memory tool that enables you to:
- Retrieve previously stored information to personalize recommendations
"""
MEMORY_ID = os.getenv("BEDROCK_AGENTCORE_MEMORY_ID")
REGION = os.getenv("AWS_REGION", "us-east-1")

# Initialize Strands telemetry
strands_telemetry = StrandsTelemetry()
strands_telemetry.setup_otlp_exporter()

app = BedrockAgentCoreApp()

def initialize_agent(context: RequestContext):
    """Initialize the agent with memory tools"""

    actor_id = 'user-20251103170743'
    session_id = 'default_session'
    if hasattr(context, 'request_headers') and getattr(context, 'request_headers'):
        actor_id = context.request_headers.get('X-Amzn-Bedrock-AgentCore-Runtime-User-Id', 'default_actor')
    if hasattr(context, 'session_id') and getattr(context, 'session_id'):
        session_id = getattr(context, 'session_id', 'default_session')
    print(f"actor: {actor_id}, session: {session_id}")
    
    model = BedrockModel(
        model_id=MODEL_ID,
    )
    provider = AgentCoreMemoryToolProvider(
        memory_id=MEMORY_ID,
        actor_id=actor_id,
        session_id=session_id,
        namespace=f"user/{actor_id}/preferences",
        region=REGION
    )
    agent = Agent(
        tools=provider.tools,
        model=model,
        system_prompt=SYSTEM_PROMPT
    )
    return agent

@app.entrypoint
def strands_agent_bedrock(payload, context: RequestContext):
    """
    Invoke the agent with a payload
    """

    agent = initialize_agent(context)

    user_input = payload.get("prompt")
    print("User input:", user_input)
    response = agent(user_input)
    return response.message['content'][0]['text']


if __name__ == "__main__":
    app.run()


## Deploying the agent to AgentCore Runtime

The `CreateAgentRuntime` operation supports comprehensive configuration options, letting you specify container images, environment variables and encryption settings. You can also configure protocol settings (HTTP, MCP) and authorization mechanisms to control how your clients communicate with the agent. 

**Note:** Operations best practice is to package code as container and push to ECR using CI/CD pipelines and IaC

In this tutorial can will the Amazon Bedrock AgentCode Python SDK to easily package your artifacts and deploy them to AgentCore runtime.

### Configure AgentCore Runtime deployment

Next we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code. 

Please note that when using the `bedrock_agentcore_starter_toolkit` to configure your agent, it takes care of the opentelemetry instrumentation. 

When configuring for containerized environment (such as docker) add the following command, an example is given below:

`CMD ["opentelemetry-instrument", "python", "runtime_agent_main.py"]`


In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "ac_rt_strands_read_ltm_obsy_demo"
response = agentcore_runtime.configure(
    entrypoint="strands_claude_ltm.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    memory_mode='NO_MEMORY',
    # disable_otel=True
)
response

### Deploy to AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

In [ ]:
import base64
from dotenv import dotenv_values

config = dotenv_values(".env")

# Langfuse configuration
# otel_endpoint = config.get("LANGFUSE_OTEL_ENDPOINT", "https://us.cloud.langfuse.com/api/public/otel")
# langfuse_secret_key = config.get("LANGFUSE_SECRET_KEY", "")  # For production key should be securely stored
# langfuse_public_key = config.get("LANGFUSE_PUBLIC_KEY", "")  # For production key should be securely stored
# langfuse_auth_token = base64.b64encode(f"{langfuse_public_key}:{langfuse_secret_key}".encode()).decode()
# otel_auth_header = f"Authorization=Basic {langfuse_auth_token}"

# Braintrust configuration
# otel_endpoint = config.get("BRAINTRUST_OTEL_ENDPOINT", "https://api.braintrust.dev/otel")
# braintrust_api_key = config.get("BRAINTRUST_API_KEY", "")  # For production key should be securely stored
# braintrust_project_id = config.get("BRAINTRUST_PROJECT_ID", "")
# otel_auth_header = f"Authorization=Bearer {braintrust_api_key}, x-bt-parent=project_id:{braintrust_project_id}"

# Bedrock AgentCore configuration
model_id = config.get("BEDROCK_MODEL_ID", "us.anthropic.claude-3-7-sonnet-20250219-v1:0")
memory_id = config.get("BEDROCK_AGENTCORE_MEMORY_ID", None)

launch_result = agentcore_runtime.launch(
    auto_update_on_conflict=True,
    env_vars={
        "BEDROCK_MODEL_ID": model_id,
        "BEDROCK_AGENTCORE_MEMORY_ID": memory_id,
        "AWS_REGION": region,
        # "OTEL_EXPORTER_OTLP_ENDPOINT": otel_endpoint,  # Use Langfuse OTEL endpoint
        # "OTEL_EXPORTER_OTLP_HEADERS": otel_auth_header,  # Add Langfuse OTEL auth header
        # "DISABLE_ADOT_OBSERVABILITY": "true",
    }
)
launch_result

### Check Deployment Status

Wait for the runtime to be ready before invoking:

In [ ]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

### Invoking AgentCore Runtime

Finally, we can invoke our AgentCore Runtime with a payload

In [ ]:
invoke_response = agentcore_runtime.invoke(
    {"prompt": "Give me restaurant recommendations in Irvine based on my food preferences"},
    # session_id="foodie-example-session-20251103170743"
)
from IPython.display import Markdown, display
display(Markdown("".join(invoke_response['response'])))

### Invoking AgentCore Runtime with boto3

Now that your AgentCore Runtime was created you can invoke it with any AWS SDK. For instance, you can use the boto3 `invoke_agent_runtime` method for it.

In [ ]:
import boto3
import yaml
import json
from IPython.display import Markdown, display

# Read agent manifest
with open('.bedrock_agentcore.yaml', 'r') as f:
    agent_manifest = yaml.safe_load(f)

agent_name = agent_manifest["default_agent"]
region = agent_manifest["agents"][agent_name]["aws"]["region"]
agent_arn = agent_manifest["agents"][agent_name]["bedrock_agentcore"]["agent_arn"]

agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=region
)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "Give me restaurant recommendations in Seattle based on my food preferences"})
)
if "text/event-stream" in boto3_response.get("contentType", ""):
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                print(line)
                content.append(line)
    display(Markdown("\n".join(content)))
else:
    try:
        events = []
        for event in boto3_response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]
    display(Markdown(json.loads("".join([s.decode("utf-8") for s in events]))))

Great! You know have a working Strands Agent capable of retrieving memories from the AgentCore Long Term Memory!

## Clean up
Let's delete the memory to clean up the resources used in this notebook.

In [ ]:
#client.delete_memory_and_wait(
#        memory_id = memory_id,
#        max_wait = 300,
#        poll_interval =10
#)